# Проєктування БД для Customer Churn · ДЗ теми 2

Мета — перевірити ER-модель, DDL PostgreSQL 16 та point-in-time (PiT) JOIN.
Контрольний набір містить **12 історичних версій для 3 клієнтів** і 6 подій.
Коректний JOIN зберігає 6 рядків; подія без доступної історії має NULL-ознаки.
Простий JOIN за ID дає 24 рядки, з них 16 використовують майбутню версію.
Модель ML тут не навчається; усі демонстраційні labels FALSE, тому це не
датасет для оцінки класифікатора.

## 1. Сценарій, джерела та припущення

Система прогнозує, чи втратить клієнт останню активну підписку в наступні
30 днів. Бізнесові сутності — клієнт, тарифний план і підписка. Підписка
має власний життєвий цикл і пов'язує рівно одного клієнта з рівно одним
планом. Клієнт і план можуть мати 0..N підписок; у часі це природний N:M,
реалізований асоціативною сутністю `subscriptions`. Додаткового штучного
bridge не потрібно. Таблиця клієнтів містить зареєстрованих людей, які ще
можуть не мати підписки чи feature snapshot.

**Походження:** [IBM Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d)
є лише референсом домену (тариф, tenure, monthly charge, churn).
Жоден рядок IBM не завантажується і не видається за історію подій.
Усі INSERT нижче — вручну задані синтетичні контрольні приклади.
`tickets_30d` не обчислений з відсутніх журналів звернень.

**Зерно:** offline — клієнт × початок версії; training — окремий момент
прогнозу; online — один матеріалізований рядок на клієнта.
`monthly_charge` — зафіксована тодішня сума, а не JOIN з нинішньою ціною плану.
`tenure_months` — значення на момент створення snapshot, не безперервний лічильник.
Label відокремлений від features і відомий лише після `label_horizon_end`;
навчання на production-даних потребувало б перевірки зрілості labels.

### Часові припущення

У цьому спрощеному наборі `valid_from` одночасно є початком чинності та
моментом доступності ознак. Пізніх подій і backfill немає. Тому перевірка
`valid_from <= prediction_ts` достатня лише в межах цього припущення.
У реальному потоці потрібні `available_at`/knowledge time або версії даних,
інакше навіть правильно записаний AS-OF може містити пізні виправлення.
Періоди напіввідкриті `[valid_from, valid_to)`; NULL зверху означає
необмежену верхню межу. Усі часові SQL-літерали мають явний UTC `+00`.

## 2. ER-модель

Позначення: `1` — обов'язково рівно один, `0..N`/`0..1` — необов'язково.
Лінії описують FK від дочірньої таблиці до батьківської. Атрибути
`valid_to` і `ended_at` дозволяють NULL; решта обов'язкові.

```text
BUSINESS
  customers (1) ──── (0..N) subscriptions (0..N) ──── (1) plans
      │                         кожна підписка має 1 customer і 1 plan
OFFLINE
      ├── (1) ──── (0..N) customer_features_offline [версії]
      └── (1) ──── (0..N) training_events [прогнози + label]
ONLINE
  customers (1) ──── (0..1) customer_features_online [lookup]
```

У таблицях і DSL нижче є **всі атрибути схеми**, типи, PK/FK та NULL.
Складений PK offline: `(customer_id, valid_from)`. Online FK є також PK,
що забезпечує максимум один materialized row на клієнта.

### Повний ER DSL (також окремий `er_diagram.mmd`)

```mermaid
erDiagram
    %% BUSINESS: each subscription has exactly one customer and exactly one plan.
    hw2_customers ||--o{ hw2_subscriptions : "has 0..N / belongs to 1"
    hw2_plans ||--o{ hw2_subscriptions : "has 0..N / uses 1"
    %% OFFLINE: each historical snapshot/event belongs to exactly one customer.
    hw2_customers ||--o{ hw2_customer_features_offline : "OFFLINE 0..N versions"
    hw2_customers ||--o{ hw2_training_events : "TRAINING 0..N events"
    %% ONLINE: a customer may have zero or one materialized row.
    hw2_customers ||--o| hw2_customer_features_online : "ONLINE 0..1 snapshot"

    hw2_customers {
        BIGINT customer_id PK
        DATE signup_date "NOT NULL"
        VARCHAR_2 country "NOT NULL"
        TEXT acquisition_channel "NOT NULL"
    }
    hw2_plans {
        INTEGER plan_id PK
        TEXT plan_name UK "NOT NULL"
        NUMERIC_8_2 monthly_price "NOT NULL"
        TEXT internet_type "NOT NULL"
        INTEGER contract_months "NOT NULL"
    }
    hw2_subscriptions {
        BIGINT subscription_id PK
        BIGINT customer_id FK "NOT NULL"
        INTEGER plan_id FK "NOT NULL"
        TIMESTAMPTZ started_at "NOT NULL"
        TIMESTAMPTZ ended_at "NULL allowed"
    }
    hw2_customer_features_offline {
        BIGINT customer_id PK,FK "NOT NULL"
        INTEGER tickets_30d "NOT NULL"
        NUMERIC_8_2 monthly_charge "NOT NULL"
        INTEGER tenure_months "NOT NULL"
        TIMESTAMPTZ valid_from PK "NOT NULL"
        TIMESTAMPTZ valid_to "NULL means unbounded"
    }
    hw2_training_events {
        BIGINT event_id PK "IDENTITY"
        BIGINT customer_id FK "NOT NULL"
        TIMESTAMPTZ prediction_ts "NOT NULL"
        TIMESTAMPTZ label_horizon_end "NOT NULL"
        BOOLEAN churn_in_horizon "NOT NULL label"
    }
    hw2_customer_features_online {
        BIGINT customer_id PK,FK "NOT NULL"
        INTEGER tickets_30d "NOT NULL"
        NUMERIC_8_2 monthly_charge "NOT NULL"
        INTEGER tenure_months "NOT NULL"
        TIMESTAMPTZ feature_ts "NOT NULL"
        TIMESTAMPTZ updated_at "NOT NULL"
    }
```

## 3. Одне локальне pgserver-середовище

Перезапустіть kernel і виконайте всі клітинки згори вниз.
Notebook самодостатній: SQL і дані вбудовані, зовнішніх файлів не потребує.
Залежності з офіційного PyPI встановлюються лише якщо відсутні.
Відомий перевірений runtime: локальний Python 3.9.25, PostgreSQL 16.2;
версії фактичного поточного запуску друкуються нижче.
Збережіть перевірку major 16:
notebook має зупинитися, якщо wheel pgserver на іншій платформі містить іншу major-версію.

Окремий тимчасовий кластер створюється тільки для цього запуску. SQL
`DROP ... CASCADE` зачіпає виключно таблиці `hw2_` цього кластера.
Усі звичайні таблиці живуть між клітинками в одному connection.
Остання клітинка закриває connection і зупиняє сервер.

In [1]:
import importlib.util
import subprocess
import sys

dependencies = {
    'pgserver': 'pgserver==0.1.4',
    'sqlalchemy': 'SQLAlchemy==2.0.54',
    'psycopg2': 'psycopg2-binary==2.9.12',
    'pandas': 'pandas==2.3.3',
}
missing = [package for module, package in dependencies.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           '--index-url', 'https://pypi.org/simple', *missing])
print('Dependencies available; Python', sys.version.split()[0])

Dependencies available; Python 3.9.25


In [2]:
import atexit
import importlib.metadata
import tempfile
from pathlib import Path
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
from sqlalchemy.exc import DBAPIError
from IPython.core.magic import register_cell_magic
from IPython.display import display

runtime_dir = tempfile.TemporaryDirectory(prefix='hw2_pg16_')
pg = pgserver.get_server(Path(runtime_dir.name) / 'data', cleanup_mode='stop')
engine = create_engine(pg.get_uri())
connection = engine.connect()

def cleanup():
    if not connection.closed:
        connection.close()
    engine.dispose()
    pg.cleanup()
    runtime_dir.cleanup()

atexit.register(cleanup)
version_num = connection.execute(text('SHOW server_version_num')).scalar_one()
assert 160000 <= int(version_num) < 170000, 'PostgreSQL 16 is required'
connection.execute(text("SET TIME ZONE 'UTC'"))
connection.commit()
print('PostgreSQL', connection.execute(text('SHOW server_version')).scalar_one())
print({p: importlib.metadata.version(p) for p in
       ['pgserver', 'SQLAlchemy', 'psycopg2-binary', 'pandas']})

sql_texts = {}
@register_cell_magic
def sql(line, cell):
    """Execute editable SQL against the single homework connection."""
    sql_texts[line.strip()] = cell
    result = connection.execute(text(cell))
    if result.returns_rows:
        records = result.mappings().all()
        frame = pd.DataFrame(records, columns=result.keys())
        globals()[line.strip()] = frame
        display(frame)
    else:
        connection.commit()
        print('SQL executed successfully')

def rows(statement):
    return [dict(r) for r in connection.execute(text(statement)).mappings()]

checks = []
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    checks.append({'check': name, 'result': 'PASS'})

PostgreSQL 16.2
{'pgserver': '0.1.4', 'SQLAlchemy': '2.0.54', 'psycopg2-binary': '2.9.12', 'pandas': '2.3.3'}


## 4. DDL та контрольні INSERT

PK/FK і доменні CHECK захищають ключі, посилання та допустимі значення.
Temporal CHECK забороняє нульовий/від'ємний інтервал, але **не міжрядкове
перекриття**. Перед training і materialization виконується окремий audit
з `assert 0`; негативний приклад нижче показує його дієвість. Для
конкурентного production-запису потрібні додаткові засоби, наприклад
exclusion constraint, а не лише цей послідовний контрольний workflow.

In [3]:
%%sql reset
-- Run only in this homework's isolated pgserver cluster; ordinary persistent tables.
SET TIME ZONE 'UTC';

DROP TABLE IF EXISTS hw2_customer_features_online, hw2_training_events,
    hw2_customer_features_offline, hw2_subscriptions, hw2_plans, hw2_customers CASCADE;


SQL executed successfully


In [4]:
%%sql ddl_1
CREATE TABLE hw2_customers (
    customer_id BIGINT PRIMARY KEY,
    signup_date DATE NOT NULL,
    country VARCHAR(2) NOT NULL CHECK (country ~ '^[A-Z]{2}$'),
    acquisition_channel TEXT NOT NULL CHECK (acquisition_channel IN ('web', 'store', 'partner'))
);


SQL executed successfully


In [5]:
%%sql ddl_2
CREATE TABLE hw2_plans (
    plan_id INTEGER PRIMARY KEY,
    plan_name TEXT NOT NULL UNIQUE,
    monthly_price NUMERIC(8,2) NOT NULL CHECK (monthly_price >= 0),
    internet_type TEXT NOT NULL CHECK (internet_type IN ('DSL', 'Fiber optic', 'None')),
    contract_months INTEGER NOT NULL CHECK (contract_months IN (1, 12, 24))
);


SQL executed successfully


In [6]:
%%sql ddl_3
CREATE TABLE hw2_subscriptions (
    subscription_id BIGINT PRIMARY KEY,
    customer_id BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    plan_id INTEGER NOT NULL REFERENCES hw2_plans(plan_id),
    started_at TIMESTAMPTZ NOT NULL,
    ended_at TIMESTAMPTZ,
    CHECK (ended_at IS NULL OR ended_at > started_at)
);


SQL executed successfully


In [7]:
%%sql ddl_4
CREATE TABLE hw2_customer_features_offline (
    customer_id BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    tickets_30d INTEGER NOT NULL CHECK (tickets_30d >= 0),
    monthly_charge NUMERIC(8,2) NOT NULL CHECK (monthly_charge >= 0),
    tenure_months INTEGER NOT NULL CHECK (tenure_months >= 0),
    valid_from TIMESTAMPTZ NOT NULL,
    valid_to TIMESTAMPTZ,
    PRIMARY KEY (customer_id, valid_from),
    CHECK (valid_to IS NULL OR valid_to > valid_from)
);
CREATE INDEX hw2_idx_features_asof
    ON hw2_customer_features_offline (customer_id, valid_from DESC);


SQL executed successfully


In [8]:
%%sql ddl_5
CREATE TABLE hw2_training_events (
    event_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    customer_id BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    prediction_ts TIMESTAMPTZ NOT NULL,
    label_horizon_end TIMESTAMPTZ NOT NULL,
    churn_in_horizon BOOLEAN NOT NULL,
    CHECK (label_horizon_end > prediction_ts)
);


SQL executed successfully


In [9]:
%%sql ddl_6
CREATE TABLE hw2_customer_features_online (
    customer_id BIGINT PRIMARY KEY REFERENCES hw2_customers(customer_id),
    tickets_30d INTEGER NOT NULL CHECK (tickets_30d >= 0),
    monthly_charge NUMERIC(8,2) NOT NULL CHECK (monthly_charge >= 0),
    tenure_months INTEGER NOT NULL CHECK (tenure_months >= 0),
    feature_ts TIMESTAMPTZ NOT NULL,
    updated_at TIMESTAMPTZ NOT NULL,
    CHECK (updated_at >= feature_ts)
);

SQL executed successfully


In [10]:
%%sql seed
-- All values are deliberately synthetic; no IBM customer records are copied.
INSERT INTO hw2_customers VALUES
    (101, DATE '2024-01-01', 'UA', 'web'),
    (202, DATE '2024-01-01', 'PL', 'store'),
    (303, DATE '2024-03-01', 'UA', 'partner');

INSERT INTO hw2_plans VALUES
    (1, 'DSL Monthly', 30.00, 'DSL', 1),
    (2, 'Fiber Annual', 50.00, 'Fiber optic', 12),
    (3, 'Fiber Premium', 70.00, 'Fiber optic', 24);

INSERT INTO hw2_subscriptions VALUES
    (1001, 101, 1, TIMESTAMPTZ '2024-01-01 00:00:00+00', TIMESTAMPTZ '2024-03-01 00:00:00+00'),
    (1002, 101, 2, TIMESTAMPTZ '2024-03-01 00:00:00+00', NULL),
    (2001, 202, 2, TIMESTAMPTZ '2024-02-01 00:00:00+00', NULL),
    (3001, 303, 3, TIMESTAMPTZ '2024-03-01 00:00:00+00', NULL);

INSERT INTO hw2_customer_features_offline
    (customer_id, tickets_30d, monthly_charge, tenure_months, valid_from, valid_to)
VALUES
    (101, 0, 30.00, 0, TIMESTAMPTZ '2024-01-01 00:00:00+00', TIMESTAMPTZ '2024-02-01 00:00:00+00'),
    (101, 1, 30.00, 1, TIMESTAMPTZ '2024-02-01 00:00:00+00', TIMESTAMPTZ '2024-03-01 00:00:00+00'),
    (101, 3, 50.00, 2, TIMESTAMPTZ '2024-03-01 00:00:00+00', TIMESTAMPTZ '2024-04-01 00:00:00+00'),
    (101, 2, 50.00, 3, TIMESTAMPTZ '2024-04-01 00:00:00+00', NULL),
    (202, 2, 50.00, 0, TIMESTAMPTZ '2024-02-01 00:00:00+00', TIMESTAMPTZ '2024-03-01 00:00:00+00'),
    (202, 1, 50.00, 1, TIMESTAMPTZ '2024-03-01 00:00:00+00', TIMESTAMPTZ '2024-04-01 00:00:00+00'),
    (202, 0, 50.00, 2, TIMESTAMPTZ '2024-04-01 00:00:00+00', TIMESTAMPTZ '2024-05-01 00:00:00+00'),
    (202, 4, 50.00, 3, TIMESTAMPTZ '2024-05-01 00:00:00+00', NULL),
    (303, 4, 70.00, 0, TIMESTAMPTZ '2024-03-01 00:00:00+00', TIMESTAMPTZ '2024-04-01 00:00:00+00'),
    (303, 2, 70.00, 1, TIMESTAMPTZ '2024-04-01 00:00:00+00', TIMESTAMPTZ '2024-05-01 00:00:00+00'),
    (303, 1, 70.00, 2, TIMESTAMPTZ '2024-05-01 00:00:00+00', TIMESTAMPTZ '2024-06-01 00:00:00+00'),
    (303, 0, 70.00, 3, TIMESTAMPTZ '2024-06-01 00:00:00+00', NULL);

-- Synthetic retrospective outcomes: no customer loses their last active
-- subscription inside these horizons. Not a classification training dataset.
INSERT INTO hw2_training_events
    (event_id, customer_id, prediction_ts, label_horizon_end, churn_in_horizon)
VALUES
    (1, 101, TIMESTAMPTZ '2024-01-15 12:00:00+00', TIMESTAMPTZ '2024-02-14 12:00:00+00', FALSE),
    (2, 101, TIMESTAMPTZ '2024-03-01 00:00:00+00', TIMESTAMPTZ '2024-03-31 00:00:00+00', FALSE),
    (3, 202, TIMESTAMPTZ '2024-01-31 23:59:59+00', TIMESTAMPTZ '2024-03-01 23:59:59+00', FALSE),
    (4, 202, TIMESTAMPTZ '2024-02-15 00:00:00+00', TIMESTAMPTZ '2024-03-16 00:00:00+00', FALSE),
    (5, 303, TIMESTAMPTZ '2024-03-31 23:59:59+00', TIMESTAMPTZ '2024-04-30 23:59:59+00', FALSE),
    (6, 303, TIMESTAMPTZ '2024-04-01 00:00:00+00', TIMESTAMPTZ '2024-05-01 00:00:00+00', FALSE);

SQL executed successfully


In [11]:
coverage = rows("""
    SELECT customer_id, COUNT(*) AS versions
    FROM hw2_customer_features_offline GROUP BY customer_id ORDER BY customer_id
""")
check('3 entities × 4 versions = 12', coverage,
      [{'customer_id': 101, 'versions': 4}, {'customer_id': 202, 'versions': 4},
       {'customer_id': 303, 'versions': 4}])
display(pd.DataFrame(coverage))

,customer_id,versions
0,101,4
1,202,4
2,303,4


## 5. Нормалізація

Бізнесові таблиці `hw2_customers`, `hw2_plans` і `hw2_subscriptions` спроєктовані у 3НФ за явно прийнятими залежностями. `customer_id` визначає дату реєстрації, країну й канал залучення; `plan_id` — назву, каталожну ціну, тип інтернету й тривалість контракту. Унікальна назва плану також є кандидатним ключем. `subscription_id` визначає клієнта, план і межі підписки. Неключові атрибути не визначають інших неключових атрибутів; залежності від частини складеного ключа тут відсутні. Підписка є природною асоціативною сутністю між клієнтами й планами, тому окрема штучна bridge-таблиця не потрібна.

Декомпозиція прибирає повторення опису плану в кожній підписці та профілі клієнта. Зміну каталогу можна внести в одному місці. Водночас історичний `monthly_charge` є зафіксованою сумою snapshot: його не відновлюємо через поточну ціну `plans`, яка могла змінитися.

Offline-таблиця має ключ `(customer_id, valid_from)` і зберігає історичні вектори; online-таблиця матеріалізує один вектор за `customer_id`. Це свідоме дублювання для читання: lookup без JOIN зменшує кількість операцій на запит. Термін «денормалізована» тут описує копіювання read model, а не автоматично порушення 3НФ: за заявленими залежностями атрибути кожного вектора визначаються його ключем.

Ризик — online-копія відстане від offline-джерела. Оновлення вибирає AS-OF версію на визначений cutoff, виконує UPSERT і зберігає `feature_ts` та `updated_at`. Перевірка свіжості відрізняє вік ознак від часу публікації; старіший snapshot не перезаписує новіший. Такий підхід прискорює доступ, але потребує узгодженого контракту обчислення й контролю оновлень.

## 6. Перекриття: обов'язковий gate перед training

Audit порівнює пари версій одного клієнта. Нерівність верхньої межі
строга: суміжні інтервали, як January→February, не перекриваються.
Порожній результат — очікуваний. `assert` зупиняє подальшу побудову, якщо
контрприклад є. Первинний ключ окремо унеможливлює рівні `valid_from`.

In [12]:
%%sql overlaps
-- Zero rows is required before building training data.
SELECT a.customer_id, a.valid_from AS a_from, a.valid_to AS a_to,
       b.valid_from AS b_from, b.valid_to AS b_to
FROM hw2_customer_features_offline AS a
JOIN hw2_customer_features_offline AS b
  ON a.customer_id = b.customer_id
 AND a.valid_from < b.valid_from
 AND a.valid_from < COALESCE(b.valid_to, 'infinity'::timestamptz)
 AND b.valid_from < COALESCE(a.valid_to, 'infinity'::timestamptz)
ORDER BY a.customer_id, a.valid_from, b.valid_from;

,customer_id,a_from,a_to,b_from,b_to


In [13]:
check('zero overlap before training', len(overlaps), 0)

## 7. Три AS-OF SELECT

`valid_from <= prediction_ts` включає початок версії. Умова
`valid_to > prediction_ts OR valid_to IS NULL` виключає її верхню межу
і підтримує поточну версію. Тому на точній межі потрапляє лише нова версія.
Label та кінець його горизонту не беруть участі у відборі features.

In [14]:
%%sql asof_single
-- AS-OF 1: customer 101 in January, before the later plan upgrade.
SELECT customer_id, tickets_30d, monthly_charge, tenure_months, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE customer_id = 101
  AND valid_from <= TIMESTAMPTZ '2024-01-15 12:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-01-15 12:00:00+00' OR valid_to IS NULL);

,customer_id,tickets_30d,monthly_charge,tenure_months,valid_from,valid_to
0,101,0,30.00,0,2024-01-01 00:00:00+00:00,2024-02-01 00:00:00+00:00


In [15]:
%%sql asof_boundary
-- AS-OF 2: exact boundary selects the new March version, never both.
SELECT customer_id, tickets_30d, monthly_charge, tenure_months, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE customer_id = 101
  AND valid_from <= TIMESTAMPTZ '2024-03-01 00:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-03-01 00:00:00+00' OR valid_to IS NULL);

,customer_id,tickets_30d,monthly_charge,tenure_months,valid_from,valid_to
0,101,3,50.00,2,2024-03-01 00:00:00+00:00,2024-04-01 00:00:00+00:00


In [16]:
%%sql training
-- AS-OF 3: preserve every training event, including an unavailable snapshot.
SELECT e.event_id, e.customer_id, e.prediction_ts,
       e.label_horizon_end, e.churn_in_horizon,
       f.tickets_30d, f.monthly_charge, f.tenure_months,
       f.valid_from AS feature_ts
FROM hw2_training_events AS e
LEFT JOIN hw2_customer_features_offline AS f
  ON f.customer_id = e.customer_id
 AND f.valid_from <= e.prediction_ts
 AND (f.valid_to > e.prediction_ts OR f.valid_to IS NULL)
ORDER BY e.event_id;

,event_id,customer_id,prediction_ts,label_horizon_end,churn_in_horizon,tickets_30d,monthly_charge,tenure_months,feature_ts
0,1,101,2024-01-15 12:00:00+00:00,2024-02-14 12:00:00+00:00,False,0.0,30.00,0.0,2024-01-01 00:00:00+00:00
1,2,101,2024-03-01 00:00:00+00:00,2024-03-31 00:00:00+00:00,False,3.0,50.00,2.0,2024-03-01 00:00:00+00:00
2,3,202,2024-01-31 23:59:59+00:00,2024-03-01 23:59:59+00:00,False,NaN,None,NaN,NaT
3,4,202,2024-02-15 00:00:00+00:00,2024-03-16 00:00:00+00:00,False,2.0,50.00,0.0,2024-02-01 00:00:00+00:00
4,5,303,2024-03-31 23:59:59+00:00,2024-04-30 23:59:59+00:00,False,4.0,70.00,0.0,2024-03-01 00:00:00+00:00
5,6,303,2024-04-01 00:00:00+00:00,2024-05-01 00:00:00+00:00,False,2.0,70.00,1.0,2024-04-01 00:00:00+00:00


In [17]:
single = rows(sql_texts['asof_single'])
boundary = rows(sql_texts['asof_boundary'])
dataset = rows(sql_texts['training'])
check('January selects old charge',
      [(r['tickets_30d'], r['monthly_charge']) for r in single], [(0, 30)])
check('exact March boundary selects new version once',
      [(r['tickets_30d'], r['monthly_charge']) for r in boundary], [(3, 50)])
# Hand-derived oracle: events 1..6; no calculation from the same join.
check('one row per training event', [r['event_id'] for r in dataset], [1,2,3,4,5,6])
check('expected tickets including absent history',
      [r['tickets_30d'] for r in dataset], [0,3,None,2,4,2])
check('historical monthly charges',
      [r['monthly_charge'] for r in dataset], [30,50,None,50,70,70])
check('snapshot tenure values',
      [r['tenure_months'] for r in dataset], [0,2,None,0,0,1])
check('no future features',
      sum(r['feature_ts'] > r['prediction_ts'] for r in dataset if r['feature_ts']), 0)
print('AS-OF checks PASS; event 3 preserved with NULL features')

AS-OF checks PASS; event 3 preserved with NULL features


Подія 3 належить уже зареєстрованому клієнту, але передує першому snapshot.
LEFT JOIN зберігає її з NULL, а не вигаданим нулем чи першою майбутньою
версією. Далі політика навчального pipeline може пропустити цю подію або
додати явний missing indicator; тут пропуск не маскується імпутацією.

## 8. Два навмисно неправильні варіанти

Entity-only JOIN повторює кожну подію чотири рази й додає майбутні версії.
Вибір `latest` за всією історією прибирає дублікати, але залишає leakage:
єдиний вибраний snapshot може бути пізнішим за прогноз. Це може завищувати
майбутню оцінку ML, але жодної оцінки моделі цей notebook не обчислює.

In [18]:
%%sql wrong_join
-- Intentional counterexample: every event receives all 4 versions.
SELECT e.event_id, COUNT(*) AS joined_rows,
       COUNT(*) FILTER (WHERE f.valid_from > e.prediction_ts) AS future_rows
FROM hw2_training_events AS e
JOIN hw2_customer_features_offline AS f ON f.customer_id = e.customer_id
GROUP BY e.event_id
ORDER BY e.event_id;

,event_id,joined_rows,future_rows
0,1,4,3
1,2,4,1
2,3,4,4
3,4,4,3
4,5,4,3
5,6,4,2


In [19]:
%%sql latest_wrong
-- Intentional counterexample: even one latest row leaks future information.
SELECT e.event_id, e.prediction_ts, f.valid_from AS wrongly_selected_feature_ts,
       f.tickets_30d, f.monthly_charge,
       f.valid_from > e.prediction_ts AS uses_future
FROM hw2_training_events AS e
LEFT JOIN LATERAL (
    SELECT valid_from, tickets_30d, monthly_charge
    FROM hw2_customer_features_offline
    WHERE customer_id = e.customer_id
    ORDER BY valid_from DESC
    LIMIT 1
) AS f ON TRUE
ORDER BY e.event_id;

,event_id,prediction_ts,wrongly_selected_feature_ts,tickets_30d,monthly_charge,uses_future
0,1,2024-01-15 12:00:00+00:00,2024-04-01 00:00:00+00:00,2,50.00,True
1,2,2024-03-01 00:00:00+00:00,2024-04-01 00:00:00+00:00,2,50.00,True
2,3,2024-01-31 23:59:59+00:00,2024-05-01 00:00:00+00:00,4,50.00,True
3,4,2024-02-15 00:00:00+00:00,2024-05-01 00:00:00+00:00,4,50.00,True
4,5,2024-03-31 23:59:59+00:00,2024-06-01 00:00:00+00:00,0,70.00,True
5,6,2024-04-01 00:00:00+00:00,2024-06-01 00:00:00+00:00,0,70.00,True


In [20]:
check('wrong entity-only join creates 24 rows', int(wrong_join.joined_rows.sum()), 24)
check('16 future pairs in wrong join', int(wrong_join.future_rows.sum()), 16)
check('latest leaks for all 6 events', int(latest_wrong.uses_future.sum()), 6)
print('Expected counterexamples reproduced; they are not training data')

Expected counterexamples reproduced; they are not training data


## 9. Негативні тести обмежень та overlap audit

Кожна помилкова вставка виконується в SAVEPOINT і відкочується.
Для рівної/зворотної межі очікується SQLSTATE `23514`, для FK `23503`,
для дублікату PK `23505`. Перекривна, але формально допустима версія
проходить CHECK; audit мусить знайти її й заблокувати training.
Після rollback вихідна історія знову має нуль перекриттів.

In [21]:
def rejected(name, statement, expected_state):
    actual = None
    try:
        with connection.begin_nested():
            connection.execute(text(statement))
    except DBAPIError as error:
        actual = error.orig.pgcode
    check(name, actual, expected_state)

prefix = 'INSERT INTO hw2_customer_features_offline VALUES '
rejected('equal temporal bounds rejected', prefix +
    "(101,0,30,0,TIMESTAMPTZ '2023-12-01 00:00:00+00',TIMESTAMPTZ '2023-12-01 00:00:00+00')", '23514')
rejected('reverse temporal bounds rejected', prefix +
    "(101,0,30,0,TIMESTAMPTZ '2023-12-02 00:00:00+00',TIMESTAMPTZ '2023-12-01 00:00:00+00')", '23514')
rejected('negative tickets rejected', prefix +
    "(101,-1,30,0,TIMESTAMPTZ '2023-12-01 00:00:00+00',TIMESTAMPTZ '2023-12-02 00:00:00+00')", '23514')
rejected('unknown customer rejected', prefix +
    "(999,0,30,0,TIMESTAMPTZ '2024-01-01 00:00:00+00',NULL)", '23503')
rejected('duplicate composite key rejected', prefix +
    "(101,0,30,0,TIMESTAMPTZ '2024-01-01 00:00:00+00',NULL)", '23505')
rejected('invalid label horizon rejected',
    "INSERT INTO hw2_training_events VALUES (99,101,TIMESTAMPTZ '2024-01-01 00:00:00+00',TIMESTAMPTZ '2024-01-01 00:00:00+00',FALSE)", '23514')

savepoint = connection.begin_nested()
try:
    connection.execute(text(prefix +
        "(101,0,30,0,TIMESTAMPTZ '2024-01-15 00:00:00+00',TIMESTAMPTZ '2024-01-20 00:00:00+00')"))
    bad_overlaps = rows(sql_texts['overlaps'])
    check('audit detects injected overlap', len(bad_overlaps), 1)
    blocked = False
    try:
        assert len(bad_overlaps) == 0, 'Training blocked: overlapping versions'
    except AssertionError:
        blocked = True
    check('overlap gate blocks training', blocked, True)
finally:
    savepoint.rollback()
check('zero overlap after rollback', len(rows(sql_texts['overlaps'])), 0)
print('Constraint and overlap negative tests PASS; no invalid rows retained')

Constraint and overlap negative tests PASS; no invalid rows retained


## 10. Online materialization і freshness

Детермінований clock — **2024-06-15 00:00:00+00**. Upsert копіює версію,
чинну на цей момент; повторний запуск дає ті самі значення. Старіший
snapshot/старіший запуск не може перезаписати новіший. `updated_at` — час
матеріалізації, `feature_ts` — вік самого snapshot: свіжий запис у БД
не робить старі features свіжими. Для демонстрації прийнятий поріг 35 днів;
це умовна політика, а не рекомендований production SLA.

In [22]:
check('zero overlap before online materialization', len(rows(sql_texts['overlaps'])), 0)

In [23]:
%%sql materialize
-- Reproducible materialization clock; never CURRENT_TIMESTAMP for fixture claims.
INSERT INTO hw2_customer_features_online
    (customer_id, tickets_30d, monthly_charge, tenure_months, feature_ts, updated_at)
SELECT customer_id, tickets_30d, monthly_charge, tenure_months, valid_from,
       TIMESTAMPTZ '2024-06-15 00:00:00+00'
FROM hw2_customer_features_offline
WHERE valid_from <= TIMESTAMPTZ '2024-06-15 00:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-06-15 00:00:00+00' OR valid_to IS NULL)
ON CONFLICT (customer_id) DO UPDATE
SET tickets_30d = EXCLUDED.tickets_30d,
    monthly_charge = EXCLUDED.monthly_charge,
    tenure_months = EXCLUDED.tenure_months,
    feature_ts = EXCLUDED.feature_ts,
    updated_at = EXCLUDED.updated_at
-- Ignore an older snapshot or an older materialization attempt.
WHERE EXCLUDED.feature_ts >= hw2_customer_features_online.feature_ts
  AND EXCLUDED.updated_at >= hw2_customer_features_online.updated_at;

SQL executed successfully


In [24]:
%%sql online_lookup
-- A bounded entity lookup and explicit freshness at the demonstration clock.
SELECT customer_id, tickets_30d, monthly_charge, tenure_months, feature_ts, updated_at,
       TIMESTAMPTZ '2024-06-15 00:00:00+00' - feature_ts AS feature_age,
       feature_ts >= TIMESTAMPTZ '2024-06-15 00:00:00+00' - INTERVAL '35 days'
           AS fresh_within_35d
FROM hw2_customer_features_online
WHERE customer_id = 101;

,customer_id,tickets_30d,monthly_charge,tenure_months,feature_ts,updated_at,feature_age,fresh_within_35d
0,101,2,50.00,3,2024-04-01 00:00:00+00:00,2024-06-15 00:00:00+00:00,75 days,False


In [25]:
before = rows('SELECT * FROM hw2_customer_features_online ORDER BY customer_id')
connection.execute(text(sql_texts['materialize']))
after = rows('SELECT * FROM hw2_customer_features_online ORDER BY customer_id')
check('online has exactly one row for each customer', len(before), 3)
check('online upsert is idempotent', after, before)
check('old snapshot flagged stale despite fresh materialization',
      rows(sql_texts['online_lookup'])[0]['fresh_within_35d'], False)
check('customer 101 feature age is 75 days',
      rows(sql_texts['online_lookup'])[0]['feature_age'].days, 75)
connection.execute(text("""
    INSERT INTO hw2_customer_features_online VALUES
    (101,99,999,99,TIMESTAMPTZ '2024-01-01 00:00:00+00',TIMESTAMPTZ '2024-06-15 00:00:00+00')
    ON CONFLICT (customer_id) DO UPDATE
    SET tickets_30d=EXCLUDED.tickets_30d, monthly_charge=EXCLUDED.monthly_charge,
        tenure_months=EXCLUDED.tenure_months, feature_ts=EXCLUDED.feature_ts,
        updated_at=EXCLUDED.updated_at
    WHERE EXCLUDED.feature_ts >= hw2_customer_features_online.feature_ts
      AND EXCLUDED.updated_at >= hw2_customer_features_online.updated_at
"""))
check('older snapshot cannot overwrite online',
      rows('SELECT * FROM hw2_customer_features_online ORDER BY customer_id'), before)
connection.commit()
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.max_colwidth', 72, 'display.width', 110):
    display(pd.DataFrame(checks))
print(len(checks), 'checks PASS')

,check,result
0,3 entities × 4 versions = 12,PASS
1,zero overlap before training,PASS
2,January selects old charge,PASS
3,exact March boundary selects new version once,PASS
4,one row per training event,PASS
5,expected tickets including absent history,PASS
6,historical monthly charges,PASS
7,snapshot tenure values,PASS
8,no future features,PASS
9,wrong entity-only join creates 24 rows,PASS


27 checks PASS


## 11. Reflection

**1. Найскладніше рішення — кардинальності.** Найскладнішим було перекласти бізнес-правила у зв’язки: клієнт може мати багато підписок, але кожна підписка має одного клієнта й один тариф. Кардинальність 1:N показує кількість можливих зв’язків; обов’язковий зовнішній ключ гарантує, що підписка посилається на наявного клієнта та наявний тариф.

**2. Компроміси моделі.** Бізнесові дані розділено на клієнтів, плани й підписки, щоб зміна тарифного опису не вимагала переписування багатьох рядків. Підписка має власний строк дії й реалізує природну асоціацію клієнтів із планами. Історичні ознаки зберігаються окремо, а online-рівень дублює чинний вектор заради читання без JOIN. Це збільшує витрати на зберігання та контроль свіжості, проте спрощує lookup. `monthly_charge` відображає минулий snapshot, тому JOIN із нинішньою ціною тарифу не замінює історії. Ознаки й labels синтетичні; однакові FALSE labels демонструють SQL, а не придатність набору для класифікації. Temporal CHECK перевіряє межі одного рядка, а PK не усуває перетини різних версій. Окремий overlap audit зупиняє підготовку training data, якщо знайдено конфлікт.

**3. Швидкий online lookup.** Для великого потоку запитів доречні попередня матеріалізація одного вектора за ID, індексований пошук і, за потреби, кешування або розподіл даних за ключем. Це концептуальні варіанти, без підтвердженої тут оцінки latency. Слід контролювати вік `feature_ts`, затримку публікації та узгодженість кешу. Вибирається версія AS-OF cutoff, а не завчасно завантажена майбутня версія.

**4. Еволюція протягом шести місяців.** Перейменування, зміна типу чи одиниць вимірювання можуть непомітно змінити вхід старої моделі. Зміна семантики `tickets_30d` потребує нової версії визначення, навіть якщо назва й тип незмінні. Видалення колонок варто відкласти до завершення підтримки залежних моделей. Потрібні версії feature-контракту, сумісні міграції та перевірка offline/online відповідності. Для пізніх подій або ретроспективних виправлень знадобиться також час доступності: нинішнє припущення про негайну доступність у `valid_from` вже не вистачатиме.

## 12. Джерела

- Чинні умови: [LMS, ДЗ теми 2](https://www.edu.goit.global/uk/learn/43680627/52381036/53713211/homework).
- [IBM Telco domain reference](https://github.com/IBM/telco-customer-churn-on-icp4d): лише сценарій, не джерело історичних рядків.
- [PostgreSQL 16: constraints](https://www.postgresql.org/docs/16/ddl-constraints.html): PK, FK, CHECK; міжрядкові умови не доводяться рядковим CHECK.
- [PostgreSQL 16: ranges](https://www.postgresql.org/docs/16/rangetypes.html): межі інтервалів і exclusion як можливий production-захист.
- [pgserver 0.1.4](https://pypi.org/project/pgserver/0.1.4/): локальний PostgreSQL runtime.

In [26]:
cleanup()
atexit.unregister(cleanup)
print('Connection closed; isolated PostgreSQL server stopped; temporary cluster removed')

Connection closed; isolated PostgreSQL server stopped; temporary cluster removed
